In [2]:
import json

In [46]:
file_name = "../client/trajectories/f4c7ea45-573e-4cb3-b5a1-bf865d171757/trajectory.json"

with open(file_name, 'r', encoding='utf-8') as file:
    data = json.load(file)

In [47]:
llaves = data["steps"][1].keys()
print(len(data["steps"]))

print(list(llaves))

17
['step', 'timestamp', 'action', 'message', 'state', 'state_hash', 'screenshot_before', 'screenshot_after']


In [60]:
def clean_record(json_data: dict):
    return_data: list = []
    steps = json_data["steps"]
    total_steps = len(steps)

    acumulador_texto: str = ""

    for i, step in enumerate(steps):
        action = step["action"]
        tipo_accion = action.get("type")
        payload = action.get("payload", {})
        
        texto_tecla = payload.get("text", "")
        nombre_tecla = payload.get("key", "")

        # --- Lógica de Teclado ---
        if tipo_accion == "KEYBOARD_EVENT":
            if nombre_tecla == "Enter" or texto_tecla == "Enter":
                # 1. Guardamos el texto acumulado (si existe)
                if acumulador_texto:
                    return_data.append({
                        "action": {
                            "type": "KEYBOARD_EVENT",
                            "payload": {"action": "text", "text": acumulador_texto}
                        },
                        "image": step["screenshot_after"]
                    })
                
                # 2. Guardamos la acción del Enter como tal
                return_data.append({
                    "action": action, # Mantenemos la acción original del Enter
                    "image": step["screenshot_after"] # Usamos el 'after' porque el Enter ya se ejecutó
                })
                
                acumulador_texto = "" 
            else:
                if texto_tecla:
                    acumulador_texto += texto_tecla
            
            continue 

        # --- Lógica de Reset para otras acciones ---
        acumulador_texto = ""

        if i < total_steps - 1:
            image = steps[i + 1]["screenshot_after"]
        else:
            image = step["screenshot_after"]

        return_data.append({
            "action": action,
            "screenshot": image
        })

    return return_data

In [61]:
for item in clean_record(data):
    print(item["action"])

{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.0193, 'position_y': 0.0277}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.1267, 'position_y': 0.091}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.7297, 'position_y': 0.1609}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.0249, 'position_y': 0.0466}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.1398, 'position_y': 0.2098}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.2353, 'position_y': 0.5105}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.2746, 'position_y': 0.5294}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'click', 'button': 'left', 'position_x': 0.0805, 'position_y': 0.485}}
{'type': 'MOUSE_EVENT', 'payload': {'action': 'dblclick', 